In [1]:
# Cell 1: imports
import json
import time
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import clear_output, display

In [2]:
# Cell 2: configuration
TRACE_PATH = Path("../data/traces/pcmb_traces.jsonl")
SOURCE_PATH = Path("../data/raw/pcmb_500.jsonl")

TOTAL_QUESTIONS = sum(1 for _ in SOURCE_PATH.open())
print(f"Target: {TOTAL_QUESTIONS} questions")

Target: 500 questions


In [3]:
# Cell 3: helpers
def load_traces():
    if not TRACE_PATH.exists():
        return pd.DataFrame()
    rows = []
    with TRACE_PATH.open() as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass  # partial write at tail — common when reading while writing
    return pd.DataFrame(rows)


def file_age_seconds():
    if not TRACE_PATH.exists():
        return None
    return time.time() - TRACE_PATH.stat().st_mtime


def progress_snapshot():
    df = load_traces()
    n_done = len(df)
    pct = 100 * n_done / TOTAL_QUESTIONS if TOTAL_QUESTIONS else 0
    age = file_age_seconds()

    print(f"=== {datetime.now().strftime('%H:%M:%S')} ===")
    print(f"Verified traces: {n_done} / {TOTAL_QUESTIONS}  ({pct:.1f}%)")
    if age is not None:
        print(f"Last write:      {age:.0f} seconds ago")
        if age > 120:
            print("   ⚠️  No writes in over 2 minutes — is the generator alive?")

    if n_done == 0:
        return df

    print(f"\nSubject breakdown:")
    print(df["subject"].value_counts().to_string())

    print(f"\nReasoning length (tokens, approx):")
    print(df["reasoning_tokens"].describe().round(0).to_string())

    # ETA
    if n_done > 5 and age is not None:
        # crude: avg time per trace = (file age since first trace) / n_done — but we don't have first trace time
        # use file ctime as a proxy for "generation started"
        started = TRACE_PATH.stat().st_ctime
        elapsed = time.time() - started
        per_trace = elapsed / n_done
        remaining = (TOTAL_QUESTIONS - n_done) * per_trace
        eta = datetime.now() + timedelta(seconds=remaining)
        print(f"\nAvg time per trace: {per_trace:.1f}s")
        print(f"ETA: {eta.strftime('%H:%M:%S')} ({timedelta(seconds=int(remaining))} from now)")

    return df


df = progress_snapshot()

=== 05:22:10 ===
Verified traces: 0 / 500  (0.0%)


In [4]:
# Cell 4: REFRESH — re-run this cell every few minutes to update
df = progress_snapshot()

if len(df) > 10:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    df["subject"].value_counts().reindex(["physics", "chemistry", "biology", "math"], fill_value=0).plot(
        kind="bar", ax=axes[0], title="Verified by subject"
    )
    axes[0].set_ylabel("count")
    axes[0].tick_params(axis="x", rotation=0)

    df["reasoning_tokens"].hist(bins=30, ax=axes[1])
    axes[1].set_title("Reasoning length distribution")
    axes[1].set_xlabel("tokens")
    plt.tight_layout()
    plt.show()

=== 05:22:42 ===
Verified traces: 0 / 500  (0.0%)
